# Session 3 · Production Python and Introduction to LLMs

Run the notebook from top to bottom with Python 3. The local exercises use temporary files. The API exercises include offline checks and separate opt-in live examples.

[Session 3 course folder](https://github.com/ReDI-Aarhus/Agentic-AI/tree/main/classes/03-production-python-intro-llms) · [Homework starter](https://github.com/ReDI-Aarhus/Agentic-AI/blob/main/classes/03-production-python-intro-llms/homework/unsolved/class3_homework_solution.ipynb)

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from types import SimpleNamespace
import os
import re
import subprocess
import sys
import venv

## Lecture · Read and write files

Read the poem in full and line by line, write and append to an output, copy the input, convert it to uppercase, write five separate items, and filter by the whole word `are`. File context managers close handles before the temporary directory is cleaned up.

In [2]:
poem = "Roses are red,\nViolets are blue,\nPython is awesome,\nAnd so are you!\n"
items = ["pen", "paper", "book", "lamp", "mug"]

with TemporaryDirectory() as folder:
    root = Path(folder)
    source = root / "input.txt"
    source.write_text(poem, encoding="utf-8")
    with source.open(encoding="utf-8") as file:
        print(file.read(), end="")
    with source.open(encoding="utf-8") as file:
        for number, line in enumerate(file, start=1):
            print(f"{number}: {line.rstrip()}")

    output = root / "output.txt"
    output.write_text("Line 1\nLine 2\nLine 3\n", encoding="utf-8")
    with source.open(encoding="utf-8") as file:
        lines = file.readlines()
    with (root / "copy.txt").open("w", encoding="utf-8") as file:
        file.writelines(lines)
    (root / "uppercase.txt").write_text("".join(lines).upper(), encoding="utf-8")
    with (root / "items.txt").open("w", encoding="utf-8") as file:
        # writelines expects the input strings to include their own separators.
        file.writelines(f"{item}\n" for item in items)
    with output.open("a", encoding="utf-8") as file:
        file.write("This is a new line!\n")
    with (root / "filtered.txt").open("w", encoding="utf-8") as file:
        file.writelines(line for line in lines if re.search(r"\bare\b", line))

    assert (root / "copy.txt").read_text(encoding="utf-8") == poem
    assert (root / "uppercase.txt").read_text(encoding="utf-8") == poem.upper()
    assert (root / "items.txt").read_text(encoding="utf-8").splitlines() == items
    assert output.read_text(encoding="utf-8").splitlines() == [
        "Line 1", "Line 2", "Line 3", "This is a new line!"
    ]
    assert (root / "filtered.txt").read_text(encoding="utf-8").splitlines() == [
        "Roses are red,", "Violets are blue,", "And so are you!"
    ]
print("All eight file tasks checked.")

Roses are red,
Violets are blue,
Python is awesome,
And so are you!
1: Roses are red,
2: Violets are blue,
3: Python is awesome,
4: And so are you!
All eight file tasks checked.


## Lecture · Project structure and dependencies

`main.py` is the entry point, `text_utils/__init__.py` marks the package, and `clean.py` is where cleaning helpers would go. This exercise creates comment-only Python files. The dependency manifest lists two packages without installing them.

In [3]:
with TemporaryDirectory() as folder:
    project = Path(folder) / "ai_project"
    package = project / "text_utils"
    package.mkdir(parents=True)
    files = {
        project / "main.py": "# Entry point for running the application.\n",
        package / "__init__.py": "# Marks text_utils as a Python package.\n",
        package / "clean.py": "# Holds helper functions for cleaning text.\n",
        project / "requirements.txt": "pandas\nnumpy\n",
    }
    for path, content in files.items():
        path.write_text(content, encoding="utf-8")
        print(path.relative_to(project))
    assert all(path.read_text(encoding="utf-8") == content for path, content in files.items())

main.py
text_utils\__init__.py
text_utils\clean.py
requirements.txt


## Lecture and homework 2 · Virtual environments

Activation changes the shell's PATH for the current terminal. Calling the environment's interpreter explicitly has the same isolation and works in a notebook. The check below creates an environment and verifies that its Python prefix differs from the base interpreter's prefix.

For the full homework, run these PowerShell commands from a local project folder. `python -m pip --version` should report a path inside `.venv`. The freeze command captures installed versions, including transitive dependencies.

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip --version
python -m pip install pandas numpy
python -c "import pandas, numpy; print(pandas.__version__, numpy.__version__)"
python -m pip freeze > requirements.txt
```

On macOS or Linux, activate with `source .venv/bin/activate`.

In [4]:
with TemporaryDirectory() as folder:
    env_dir = Path(folder) / ".venv"
    # Installing packages is a separate terminal step.
    venv.EnvBuilder(with_pip=False).create(env_dir)
    executable = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    validation = subprocess.run(
        [str(executable), "-c", "import sys; print(sys.prefix != sys.base_prefix)"],
        check=True,
        capture_output=True,
        text=True,
    )
    assert validation.stdout.strip() == "True"
    print("Virtual environment interpreter verified.")

Virtual environment interpreter verified.


## Lecture and homework 1, 3.1, and 3.2 · Import a function and handle errors

The starter places the division function in `utils/utils.py`. A short script imports it from that package and checks that 6 divided by 3 produces 2. The completed function catches exactly the two exceptions required by the homework. Zero divisors and incompatible values such as strings return `None`. The notebook generates the package in a temporary directory to keep the submission in one file.

In [5]:
division_module = """def utils_divide(a, b):
    try:
        return a / b
    except (ZeroDivisionError, TypeError):
        # These are the two failures covered by the homework contract.
        return None
"""

with TemporaryDirectory() as folder:
    project = Path(folder)
    package = project / "utils"
    package.mkdir()
    (package / "__init__.py").write_text("", encoding="utf-8")
    (package / "utils.py").write_text(division_module, encoding="utf-8")
    check = "from utils.utils import utils_divide; assert utils_divide(6, 3) == 2"
    subprocess.run([sys.executable, "-c", check], cwd=project, check=True)

    # Isolate the example import from any package named utils in the notebook.
    from importlib.util import module_from_spec, spec_from_file_location

    spec = spec_from_file_location("session_03_division", package / "utils.py")
    division = module_from_spec(spec)
    spec.loader.exec_module(division)
    utils_divide = division.utils_divide

assert utils_divide(10, 2) == 5
assert utils_divide(10, 0) is None
assert utils_divide("a", 5) is None
print("Package import and both exception cases passed.")

Package import and both exception cases passed.


## Homework 4 · Load numbers from a dotenv file

Dotenv values are text, so convert both values to numbers before dividing. Install the parser in the notebook environment with `python -m pip install python-dotenv`. The check reads the two values directly from a temporary .env file without changing the kernel's environment.

In [6]:
from dotenv import dotenv_values


def divide_configured_numbers(values):
    try:
        first = float(values["NUMBER1"])
        second = float(values["NUMBER2"])
    except (KeyError, ValueError) as error:
        raise ValueError("Set NUMBER1 and NUMBER2 to numeric values.") from error
    return utils_divide(first, second)


with TemporaryDirectory() as folder:
    dotenv_sample = Path(folder) / ".env"
    dotenv_sample.write_text("NUMBER1=10\nNUMBER2=2\n", encoding="utf-8")
    # Read the sample without modifying the current process environment.
    values = dotenv_values(dotenv_sample)
    result = divide_configured_numbers(values)

assert result == 5
assert divide_configured_numbers({"NUMBER1": "10", "NUMBER2": "0"}) is None
print(f"Configured division: {result:g}")

Configured division: 5


## Lecture · Compare how models follow prompts

Use the same prompts for both models and check the response against the assumptions below. The bridge prompt explicitly allows everyone to cross together. For the probability question, a six-sector spinner illustrates the difference between advancing to a known position and sampling again.

| Prompt | Expected check |
|---|---|
| Face west in London. Is Edinburgh to your left or right? | Right, because north is to your right. |
| Four people cross together and take 1, 2, 5, and 10 minutes individually. How long for the group? | 10 minutes at the slowest person's pace. |
| Only one of six spinner sectors is winning and you are on it. Is advancing one sector or spinning again better? | Advancing gives zero chance of winning. A fresh uniform spin gives a one-in-six chance. |
| A person and goat have a boat that fits both. How can they cross? | They ride together. |
| Write ten sentences ending with the word apple. | Count ten sentences and check that the last word of each is apple. |
| Five shirts dry in five hours. With the same conditions and enough space, how long for 30 shirts? | Five hours, because they dry in parallel. |

In [7]:
prompts = [
    "Face west in London. Is Edinburgh to your left or right? Explain briefly.",
    "Four people cross a bridge together. Individually they take 1, 2, 5, and 10 minutes. How long does the group take?",
    "A six-sector spinner has one winning sector and is on it. Is advancing one sector or spinning again better? Give the probabilities.",
    "A person and a goat have a boat that fits both. How can they cross?",
    "Write exactly ten short sentences ending with the word apple. Put one sentence on each line.",
    "Five shirts dry in five hours. With the same weather and enough space, how long for 30 shirts to dry?",
]


def follows_apple_instruction(answer):
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    return len(lines) == 10 and all(re.search(r"\bapple[.!?]?$", line, re.I) for line in lines)


assert follows_apple_instruction("\n".join(f"Sentence {i} ends with apple." for i in range(1, 11)))
assert not follows_apple_instruction("I ate an apple.")
print(f"{len(prompts)} comparison prompts prepared.")

6 comparison prompts prepared.


## Lecture and homework 5 · OpenAI and Groq APIs

Install the SDKs in the notebook's environment with `python -m pip install python-dotenv openai groq`. Use a local `.env`, or set `COURSE_ENV_FILE` to the path of your existing private dotenv file before starting Jupyter. Git ignores `.env` and its variants. For Azure OpenAI v1, set `OPENAI_ENDPOINT` to your resource's `/openai/v1/` base URL and use `AZURE_OPENAI_DEPLOYMENT` as the model name.

```dotenv
NUMBER1=10
NUMBER2=2
OPENAI_API_KEY=
OPENAI_MODEL=
OPENAI_ENDPOINT=
AZURE_OPENAI_DEPLOYMENT=
GROQ_API_KEY=
GROQ_MODEL=openai/gpt-oss-20b
GROQ_COMPARE_MODELS=
```

For the OpenAI exercise, choose a model that supports the sampling parameters being tested. Temperature changes sampling randomness, top-p limits the probability mass considered, and the completion token cap limits output length. The examples vary one setting at a time. For Groq, list the available model IDs, ask one question, then optionally compare two models by setting `GROQ_COMPARE_MODELS` to two comma-separated IDs.

The offline check verifies request construction and response parsing. The live cell sends only the selected exercises after their switches are enabled.

[OpenAI Chat API](https://developers.openai.com/api/reference/resources/chat) · [Groq quickstart](https://console.groq.com/docs/quickstart)

**Live verification.** Azure OpenAI returned responses for all four text trials and correctly identified red in the optional image test. The Groq model-list request returned HTTP 401, so the Groq completion and model comparison remain unverified.

The outputs below are saved from an authenticated live run. The switches default to off so opening or rerunning the public notebook does not automatically send requests.

In [8]:
def load_course_environment():
    from dotenv import load_dotenv

    load_dotenv(Path(os.environ.get("COURSE_ENV_FILE", ".env")).expanduser(), override=False)


def openai_client_and_model():
    from openai import OpenAI

    endpoint = os.environ.get("OPENAI_ENDPOINT")
    options = {"base_url": endpoint.rstrip("/") + "/"} if endpoint else {}
    model = os.environ.get("AZURE_OPENAI_DEPLOYMENT") if endpoint else None
    model = model or os.environ["OPENAI_MODEL"]
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"], **options), model


def ask_model(client, model, question, **settings):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        **settings,
    )
    return response.choices[0].message.content


openai_trials = [
    ("Baseline", {"temperature": 0.2, "top_p": 1.0, "max_completion_tokens": 150}),
    ("Higher temperature", {"temperature": 0.8, "top_p": 1.0, "max_completion_tokens": 150}),
    ("Lower top-p", {"temperature": 0.2, "top_p": 0.6, "max_completion_tokens": 150}),
    ("Shorter completion", {"temperature": 0.2, "top_p": 1.0, "max_completion_tokens": 60}),
]
captured = []


def demo_completion(**request):
    captured.append(request)
    return SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(content="Sample response for the offline test."))]
    )


demo_client = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(create=demo_completion)))
label, settings = openai_trials[0]
assert ask_model(demo_client, "demo-model", prompts[0], **settings).startswith("Sample response")
assert captured[0]["messages"] == [{"role": "user", "content": prompts[0]}]
assert captured[0]["model"] == "demo-model"
assert all(captured[0][name] == value for name, value in settings.items())
print("Offline API request and response checks passed.")

Offline API request and response checks passed.


In [9]:
RUN_OPENAI = False
RUN_GROQ = False
RUN_GROQ_COMPARISON = False

if RUN_OPENAI or RUN_GROQ or RUN_GROQ_COMPARISON:
    load_course_environment()

if RUN_OPENAI:
    openai_client, openai_model = openai_client_and_model()
    question = "Explain what a Python virtual environment does in two sentences."
    for label, settings in openai_trials:
        print(label, ask_model(openai_client, openai_model, question, **settings))

if RUN_GROQ or RUN_GROQ_COMPARISON:
    from groq import Groq

    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
    available = {model.id for model in groq_client.models.list().data}
    print("Available Groq models:", ", ".join(sorted(available)))

if RUN_GROQ:
    groq_model = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")
    if groq_model not in available:
        raise ValueError("Choose GROQ_MODEL from the listed IDs.")
    print(ask_model(groq_client, groq_model, "Why should Python projects use a virtual environment?"))

if RUN_GROQ_COMPARISON:
    models = [model.strip() for model in os.environ.get("GROQ_COMPARE_MODELS", "").split(",") if model.strip()]
    if len(set(models)) != 2 or len(models) != 2 or any(model not in available for model in models):
        raise ValueError("Set GROQ_COMPARE_MODELS to two distinct IDs from the list.")
    for model in models:
        print(f"\n{model}")
        for question in prompts:
            answer = ask_model(groq_client, model, question)
            print(f"Prompt: {question}\nAnswer: {answer}")
            if question == prompts[4]:
                print("Apple constraint passed:", follows_apple_instruction(answer or ""))

if not (RUN_OPENAI or RUN_GROQ or RUN_GROQ_COMPARISON):
    print("Live API calls skipped. The offline checks above ran without credentials.")

Baseline
A Python virtual environment creates an isolated space for a project so it can use its own Python interpreter and installed packages without affecting other projects or the system Python. This helps avoid dependency conflicts and makes it easier to manage and reproduce a project’s setup.

Higher temperature
A Python virtual environment creates an isolated space for a project with its own Python interpreter and installed packages, separate from the system-wide Python setup. This lets you manage dependencies for each project independently and avoid version conflicts.

Lower top-p
A Python virtual environment creates an isolated space for a project with its own Python interpreter and installed packages, separate from the system-wide Python setup. This lets you manage dependencies for each project independently and avoid version conflicts.

Shorter completion
A Python virtual environment creates an isolated space for a project with its own Python interpreter and installed packages

### Optional · Image input

With the same OpenAI client, send a text question and an `image_url` content part to a model that supports images. A public image URL or a base64 data URL is accepted by the Chat API. This example uses a public URL supplied through `OPENAI_IMAGE_URL` and runs only when enabled. [OpenAI Chat API](https://developers.openai.com/api/reference/resources/chat)

The saved example response used a generated red square.

In [10]:
RUN_OPENAI_IMAGE = False

if RUN_OPENAI_IMAGE:
    load_course_environment()
    client, model = openai_client_and_model()
    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe the main objects in this image."},
                {"type": "image_url", "image_url": {"url": os.environ["OPENAI_IMAGE_URL"]}},
            ],
        }],
        max_completion_tokens=150,
    )
    print(response.choices[0].message.content)

The image appears to be a solid bright red color with no distinct objects visible.
